# Notebook 10.3 — Conversation History & Evidence Validation

## Objective
I validate the evidence/citation contract and prepare a lightweight conversation log. A good assistant should preserve source IDs, keep answer claims grounded, and make the interaction auditable.

## Notebook presentation rule
I place conclusions after evidence-validation cells that produce substantive analytical output.

In [1]:
import sys
from pathlib import Path
import re
import pandas as pd

sys.path.insert(0, str(Path('../app').resolve()))
from query_engine import answer_query, retrieve

print_result = lambda message: print(f'Result: {message}')


In [2]:
test_question = "How does a city churn rate compare with its peer-city benchmark?"
result = answer_query(test_question, mysql_password=None, top_k=5)
retrieved_ids = set(result["retrieved"]["evidence_id"].tolist())
answer = result["answer"]
citations = set(re.findall(r"\[(?:SQL)?\d+\]", answer))

print("Question:", test_question)
print("Answer:\n", answer[:3000])
print("Citations found in answer:", sorted(citations))
print("Retrieved evidence IDs:", sorted(retrieved_ids))
print_result(f"Found {len(citations)} citation tokens in the generated answer.")


Question: How does a city churn rate compare with its peer-city benchmark?
Answer:
 The evidence is insufficient to determine how a city's churn rate compares with its peer-city benchmark. 

While the evidence provides the peer-city benchmark value (26.62% median across cities with at least 25 customers, evidence ID [4]), it does not contain the observed churn rate for any specific city that would allow for comparison. 

The evidence defines the benchmark gap calculation as `Benchmark Gap = Observed Churn Rate − Benchmark Churn Rate` (evidence IDs [1] and [2]), but no observed city churn rate values are provided in the evidence. Without the observed churn rate for a specific city, I cannot calculate or describe the comparison.

The evidence states that "Geographic patterns are descriptive and do not prove that geography causes churn" and "Benchmark gaps are descriptive comparisons and are not predictive forecasts" (evidence ID [1]), but this does not help with the specific comparison q

### Result & conclusion
The citation audit extracts evidence IDs from the generated answer and compares them with the IDs actually supplied to the model. This catches citation references that do not exist in the retrieved context.

In [3]:
def validate_citations(answer: str, retrieved_ids: set[str]) -> dict:
    citations = set(re.findall(r'\[(?:SQL)?\d+\]', answer))
    unknown = sorted(citations - retrieved_ids)
    return {
        'citation_count': len(citations),
        'unknown_citations': unknown,
        'all_citations_known': not unknown,
    }

validation = validate_citations(result['answer'], retrieved_ids)
display(pd.DataFrame([validation]))
print_result('Citation validation passes when every cited evidence ID is present in the supplied context.')

,citation_count,unknown_citations,all_citations_known
0,3,[],True


Result: Citation validation passes when every cited evidence ID is present in the supplied context.


### Result & conclusion
I now have a deterministic citation-validation check. It validates citation identity, not whether the cited evidence semantically supports every sentence; that still requires content review.

In [4]:
conversation = []
turns = [
    'What is revenue at risk in RetainIQ?',
    'What does a 20 percent recovery scenario mean?',
    'Which customer segment has the highest revenue at risk?'
]

for i, q in enumerate(turns, start=1):
    r = answer_query(q, mysql_password=None, top_k=5)
    conversation.append({
        'turn': i,
        'question': q,
        'mode': r['mode'],
        'answer': r['answer'],
        'evidence_ids': '|'.join(r['retrieved']['evidence_id'].tolist()),
        'top_sources': '|'.join(r['retrieved']['source_name'].head(3).tolist()),
        'citation_valid': validate_citations(r['answer'], set(r['retrieved']['evidence_id']))['all_citations_known'],
    })

conversation_df = pd.DataFrame(conversation)
conversation_df.to_csv('../outputs/conversation_validation_log.csv', index=False)
display(conversation_df[['turn','mode','question','evidence_ids','citation_valid']])
print_result(f'Prepared {len(conversation_df)} conversation turns for audit and saved them to conversation_validation_log.csv.')

,turn,mode,question,evidence_ids,citation_valid
0,1,llm,What is revenue at risk in RetainIQ?,[1],True
1,2,llm,What does a 20 percent recovery scenario mean?,[1]|[2],True
2,3,llm,Which customer segment has the highest revenue...,[1]|[2]|[3]|[4]|[5],True


Result: Prepared 3 conversation turns for audit and saved them to conversation_validation_log.csv.


### Result & conclusion
The conversation log keeps the question, answer mode, evidence IDs, source metadata, and citation-validity flag together. This creates an auditable history for later app logging.

## Grounding rule
If evidence is insufficient, the assistant should say so. A complete citation ID check is necessary but not sufficient; human review of important answers remains part of the quality process.

## Handoff
The final notebook moves this validated backend into the Streamlit interface.